# P20 — Anticiper les secteurs européens qui dépasseront leurs quotas carbone (EU ETS)

**Contexte métier.** Le marché carbone européen (EU ETS) couvre ~10 000 installations : chaque année, chaque secteur/pays reçoit des quotas (allocation) et déclare ses émissions vérifiées. Dépasser son allocation oblige à acheter des quotas au prix du marché ; anticiper ce déficit est une décision financière pour les industriels, les traders et les régulateurs.

**Objectif de l'analyse exploratoire.** Construire un panel pays x secteur x année et identifier la cible (émissions > allocation) et les régresseurs (pays, secteur, historique).

**Données.** EEA « EU ETS data viewer » — base septembre 2026, série 2005-2025, 90 325 lignes au format long.

## 1. Téléchargement

Export officiel de la base EU ETS de l'EEA (zip, xlsx ~3,6 Mo).

In [ ]:
import os, urllib.request, zipfile

os.makedirs("data", exist_ok=True)
URL = "https://sdi.eea.europa.eu/datashare/s/aZ7Ti5HkRGEbF62/download"
if not os.path.exists("data/ets_v03.zip"):
    urllib.request.urlretrieve(URL, "data/ets_v03.zip")
with zipfile.ZipFile("data/ets_v03.zip") as z:
    z.extractall("data")
print("OK", os.path.getsize("data/ets_v03.zip") // (1024 * 1024), "Mo")

## 2. Chargement

Une ligne = pays x secteur x année x ligne comptable (allocation, émissions vérifiées, unités restituées...). Attention : la colonne année contient aussi des lignes de totaux (texte) à écarter.

In [ ]:
import pandas as pd

df = pd.read_excel("data/eea_t_eu-emission-trading-scheme_p_2005-2025_v03_r00/ETS_Database_September_2026.xlsx",
                   engine="openpyxl")
print(df.shape, "lignes x colonnes")
print(df["citl_information"].value_counts().head(8).to_string())

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df[df["year"].notna()].copy()
df["year"] = df["year"].astype(int)
print("Lignes avec année valide :", len(df))

## 3. Analyse

Pivot en panel (pays x secteur x année) : allocation (ligne 1) contre émissions vérifiées (ligne 2), puis part des couples sous-alloués.

In [ ]:
keys = ["country_code", "main_activity_code", "year"]
val = df.pivot_table(index=keys, columns="citl_information", values="value", aggfunc="first")
val.columns = [str(c) for c in val.columns]
alloc = "1. Total allocated allowances (EUA or EUAA)"
emis = "2. Verified emissions"
p = val[[alloc, emis]].dropna()
p.columns = ["allocation", "emissions"]
p = p[(p["allocation"] > 0) & (p["emissions"] > 0)]
print("Paires secteur-pays-année :", len(p))
print("Part sous-allocation (émissions > allocation) :",
      round(100 * (p["emissions"] > p["allocation"]).mean(), 1), "%")
p["ratio"] = p["emissions"] / p["allocation"]
print(p["ratio"].describe().round(2).to_string())
import numpy as np
print("Corrélation log allocation vs log émissions :",
      round(np.log10(p["allocation"]).corr(np.log10(p["emissions"])), 2))
print()
print("Émissions vérifiées par année (Mt) :")
print(p.reset_index().groupby("year")["emissions"].sum().div(1e6).round(1)
      .loc[[2005, 2010, 2015, 2020, 2024]].to_string())

## 4. Résultats observés (exécution du 16/10/2026)

- base brute : 90 325 lignes x 9 colonnes ; 36 pays, 31 codes d'activité, années 2005-2025
- pivot panel : 11 761 lignes pays x secteur x année, 19 lignes comptables
- paires allocation/émissions exploitables : 9 175
- part sous-allocation (émissions > allocation) : 48,8 % — problème équilibré
- ratio émissions/allocation : médiane 0,99, Q1 0,78, Q3 1,26 ; quelques valeurs extrêmes (max 1 725, très petits secteurs)
- corrélation log-log allocation vs émissions : 0,96
- émissions vérifiées du panel : 4 574 Mt (2005) → 4 244 Mt (2015) → 3 261 Mt (2020) → 2 568 Mt (2024) : baisse structurelle
- à noter : la ligne 2.1b « expected, gap-filled » comble la dernière année (996 lignes) — 2025 n'est pas définitive

## 5. Variable cible et régresseurs

**Cible (classification binaire) : « le couple pays-secteur dépasse son allocation » pour une année donnée** (émissions vérifiées > allocation) — 48,8 % de positifs. Variante régression : log du ratio émissions/allocation.

**Régresseurs retenus (pour anticiper l'année N) :**
- `country_code`, `main_activity_code` (combustion, raffinage, ciment, sidérurgie...) : effets dominants
- historique du couple : émissions, allocation et ratio des années N-1, N-2 (forte inertie industrielle)
- `year` : tendance de fond (-45 % d'émissions depuis 2005)
- l'allocation N quand elle est publiée avant la déclaration

**Pièges :**
- la question métier est l'anticipation : ne pas entraîner sur les features de l'année N (émissions vérifiées N) pour prédire le dépassement N — c'est une fuite. Utiliser l'historique N-1
- la corrélation log-log 0,96 : l'allocation est calée sur les émissions passées — le modèle doit battre la baseline « ratio N-1 persiste »
- ratios extrêmes (max 1 725) sur de très petits secteurs : winsoriser ou pondérer
- 2025 gap-filled : l'écarter du test ou la marquer
- les totaux textuels dans la colonne année doivent être exclus (to_numeric, errors='coerce')